In [129]:
import numpy as np
import random
from tqdm.auto import tqdm

In [130]:
x = [
    [1, 2, 3]
]

y = [3,4,5]

# convert normal python array into numpy ndarray
x = np.array(x)

PROBLEM_SIZE  = np.shape(x)[0]
PROBLEM_SIZE

1

In [131]:
max_coefficient = 10 # adjustable depending on output scale

def compute_coefficient():
    """returns a random coefficient between 1 and maximum coefficient"""
    return np.random.randint(1, max_coefficient)

In [132]:
# print(np.divide(5,2))
# print(np.remainder(5,2))
# print(np.pow(5,2))

# a x b = b x 1  ->  a x 1
# 3 x 3 * 3 x 
# y = np.add(np.array(x[0]), 5)
print(y)

# print(np.array(x).shape)

[3, 4, 5]


In [133]:
binary_operators = [np.add, np.subtract, np.dot, np.divide, np.pow]

BINARY_OPERATORS = {
    "+": np.add,
    "-": np.subtract,
    "*": np.dot,
    "/": np.divide,
    "^": np.pow
}

#https://numpy.org/doc/2.1/reference/routines.math.html
UNARY_OPERATORS = {
        "": lambda x: x,  
        "sin": np.sin,
        "cos": np.cos,
        "tan":np.tan,
        "log": np.log,
        "exp": np.exp,
        "arccos": np.arccos,
        "arcsin":np.arcsin,
        "arctan":np.arctan,
        "sqrt":np.sqrt,
        "cbrt":np.cbrt,
        "square":np.square,
        "abs":np.abs,
        "reciprocal":np.reciprocal
    }

#VARIABLES = [f"X_{i}" for i in range(PROBLEM_SIZE)]

#VARIABLES_WEIGHTS = [[1/len(VARIABLES) for _ in range(len(VARIABLES))]]
VARIABLES_MAP = {f"X_{i}": x[i] for i in range(PROBLEM_SIZE)}    # {'X_0': [1, 2, 3], 'X_1': [4, 5, 6], 'X_2': [7, 8, 9]}
LEAVES = [i for i in range(10)] + list(VARIABLES_MAP.keys())
print(VARIABLES_MAP)
print(LEAVES)

{'X_0': array([1, 2, 3])}
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 'X_0']


## Tree structure

In [134]:
class TreeNode:
    def __init__(self, value):
        self.value = value      # This can be an operator or operand
        self.left = None        # Left child
        self.right = None       # Right child
        self.coefficient = None # multiplicative coefficient for a variable

def validate_tree(node):
    if not node:
        return True
    
    if node.value in BINARY_OPERATORS:
        if not node.left or not node.right:
            return False  # Operators must have two children
        return validate_tree(node.left) and validate_tree(node.right)
    
    elif node.value in UNARY_OPERATORS:  # Allow unary operators
        if node.right and not node.left:
            return validate_tree(node.right) 
        return False  # Unary operators must have one child on the right
    
    # elif node.value in VARIABLES_MAP and isinstance(node.value, str):  # Allow variables
    elif node.value in LEAVES:
        return True
    else:
        return False  # Invalid value
    
    
# (3 + 2) * (4 + 5)        Treenode (value = *, left = Treenode (value = +, left = 3, right = 2), right = Treenode (value = +, left = 4, right = 5))
def evaluate_tree(node):
    if not node:
        raise ValueError("Cannot evaluate an empty tree.")
    
    # Check if it's a binary operator
    if node.value in BINARY_OPERATORS:
        left_val = evaluate_tree(node.left)
        right_val = evaluate_tree(node.right)
        return BINARY_OPERATORS[node.value](left_val, right_val)
    
    # Check if it's a unary operator
    elif node.value in UNARY_OPERATORS:
        right_val = evaluate_tree(node.right)  # Typically applies to right child
        return UNARY_OPERATORS[node.value](right_val)  # Correct unary application
    
    # Check if it's a variable
    elif node.value in VARIABLES_MAP:
        return VARIABLES_MAP[node.value]  # Lookup the variable value
    
    # Check if it's a numeric constant or coefficient
    elif isinstance(node.value, (int, float)):
        return node.value  # Return as-is for numeric leaf nodes
    
    # If none of the above, it's an error
    else:
        raise ValueError(f"Invalid node value: {node.value}")

def are_compatible(operator, node):
    return True

def random_initial_tree(depth, maxdepth, variables, binary_operators, unary_operators):
    if depth == maxdepth:  # Add a variable until they are all chosen, if yes add a number
        if len(variables):
            var = random.choice(variables)
            leaf = TreeNode(var)
            leaf.coefficient = compute_coefficient()
            variables.remove(var)
        else:
            leaf = TreeNode(compute_coefficient())
            leaf.coefficient = 1
        return leaf
    
    if depth == maxdepth - 1: # Add a unary operator
        node = TreeNode(None)
        node.right = random_initial_tree(depth + 1, maxdepth, variables, binary_operators, unary_operators)
        node.left = None
        if np.random.choice([0, 1]): # 50% chance of invariant unary operator, 50% chance of any of the other unary operators
            node.value = TreeNode("")
        else:
            available_unary = [op for op in unary_operators if are_compatible(op, node.right)]
            node.value = TreeNode(np.random.choice(available_unary)) # If a choice of a variant unary operator was made, choose a random variant from all the possible ones
        return node
    
    else: # Add a binary operator
        node = TreeNode(np.random.choice(list(BINARY_OPERATORS.keys())))
        node.left = random_initial_tree(depth + 1, maxdepth, variables, binary_operators, unary_operators)
        node.right = random_initial_tree(depth + 1, maxdepth, variables, binary_operators, unary_operators)
        return node


### printing functions

In [135]:
def print_tree(node: TreeNode, is_root: bool = True):
    """
    Prints the symbolic representation of the tree with the names of the variables
    
    Args:
        node: TreeNode
        is_root: bool
    """
    if not node:
        return

    # Add parentheses around subexpressions unless it's the root
    if not is_root:
        print("(", end="")

    # Traverse the left child
    if node.left:
        print_tree(node.left, is_root=False)

    # Print the current node's value
    print(node.value, end=" ")

    # Traverse the right child
    if node.right:
        print_tree(node.right, is_root=False)

    # Close parentheses if not the root
    if not is_root:
        print(")", end="") 


def print_tree_values(node: TreeNode, is_root: bool = True):
    """ 
    Prints the symbolic representation of the tree with the values of the variables
    
    Args:
        node: TreeNode
        is_root: bool
    """
    if not node:    
        return

    # Add parentheses around subexpressions unless it's the root
    if not is_root:
        print("(", end="")

    # Traverse the left child
    if node.left:
        print_tree_values(node.left, is_root=False)

    # Print the current node's value
    print(VARIABLES_MAP[node.value] if node.value in VARIABLES_MAP else node.value, end=" ")

    # Traverse the right child
    if node.right:
        print_tree_values(node.right, is_root=False)

    # Close parentheses if not the root
    if not is_root:
        print(")", end="") 


def print_expr(node):
    """
    Prints the symbolic representation of the tree with the names of the variables inside an expression
    """
    print_tree(node) 
    print("= y")

def print_expr_values(node):
    """
    Prints the symbolic representation of the tree with the values of the variables inside an expression
    """
    print_tree_values(node) 
    print(" = ", end="")
    print(evaluate_tree(node))

In [136]:
def generate_initial_solution():
    variables = list(VARIABLES_MAP.keys())
    n_variables = len(variables)
    n_leaves = int(2 ** np.ceil(np.log2(n_variables)))
    n_actual_leaves = n_leaves * 2
    binary_operators = list(BINARY_OPERATORS.keys())
    unary_operators = list(UNARY_OPERATORS.keys())
    unary_operators.remove("")
    max_depth = np.log2(n_actual_leaves)

    while True:
        variables = list(VARIABLES_MAP.keys())
        root = random_initial_tree(0, max_depth, variables, binary_operators, unary_operators)
        try:
            print_expr(root)
            if validate_tree(root):
                evaluate_tree(root)
                return root
        except:
            pass
            


## steps
- Generate random tree
    - we need each variable at least once 
    - each variable has exactly one coefficient chosen as a random float number in the range [?, ?]
    - each variable has exactly one unary operator
    - unary operator is chosen as: 50% chance of "" (i.e. no change to the variable), 50% chance of choosing among all other unary operators
        - check if the unary operator is appliable to the variable ->
            ```
            leaves_map = {}
            for e in leaves:
                available_unary_operators = [op for op in list(UNARY_OPERATORS.keys()) if op.is_applicable(e)]
                chosen_unary_operator = 50% chance of "" (i.e. no change to the variable), 50% chance of choosing among [available_unary_operators]
                leaves_map[e] = [chosen_unary_operator]
            # leaves = [-2, 3]
            # leaves_map = {-2: square, 3: log}
            for e in leaves:
                node = leaves_map[e]
                node.left = null
                node.right = e
                # insert node to tree
            ```
    - number of leaves = nearest power of two greater than keys.length()
    - number of actual leaves = [number of leaves] * 2
    - number of coefficients = [number of leaves] - keys.length()
    - number of binrary operators = total number of nodes in  tree with [number of leaves] leaves - [number of leaves]]
    - validate tree
    - if valid, return tree
    - else, ?
- Example:
    - x.length() = 3
    - number of leaves = 4
    - number of actual leaves = 8
    - number of coefficients = 1
    - number of operands = 3

    ```bash
                    +
            /                  \
            *                    +
        /      \           /        \
      u        1          1          u
    /   \    /   \      /   \       /  \
    nul  *  nul   *    nul    *     nul *
    ```


In [ ]:
root = TreeNode("+")
root.left = TreeNode(3)
root.right = TreeNode("*")
root.right.left = TreeNode(4)
root.right.right = TreeNode("X_0") # 3 + 4x = 3 + 4 x[1]
#(3 + ((3 + x) * x))
# 0     +
#     / \
#1   3   *
#       / \
#2      /   x
#     /
# 3   +
#   / \
# 4 3   x

print_expr(root)
print("--------------------------")
print_expr_values(root)

# cos((5 * 4) - square(3))

#      cos
#       |
#       -
#      / \
#     *   square
#    / \    /
#   5   4  3

#binary tree with depth 2
#             +
#       /            \
#      *                +
#     / \           /        \
#   u     1          1          u
# / \     / \       / \        / \
#nul x0  nul x2     nul c     nul x1
# depth = 3 partendo da 0

# Initial solution : complete binary tree
#7 keys -> 8 leaves -> 1 coefficient



new_root = TreeNode("cos")
new_root.right = TreeNode("-")
new_root.right.left = TreeNode("*")
new_root.right.left.left = TreeNode(5)
new_root.right.left.right = TreeNode(4)
new_root.right.right = TreeNode("square")
new_root.right.right.right = TreeNode(3)

print(validate_tree(new_root))
print_expr(new_root)
print("\n--------------------------")
#print_expr_values(new_root)


print("--------------------------")
random_root = generate_initial_solution()
print_expr(random_root)
if validate_tree(random_root):
    print("valid")
    print_expr_values(random_root)

In [16]:
def mse(x,y):
    return (x-y)**2

In [ ]:
# initial_solution = TreeNode(operators[random.randint(0,num_operators)])
# print(initial_solution.value)
# initial_solution.left = TreeNode(x[0])
# initial_solution.right = TreeNode(random.randint(1,10))
# print(initial_solution.right.value)
# print(evaluate_tree(initial_solution,{}))

# # y = x+ n
# # mse(evaluate_tree(initial_solution, {}), y[0])
# # tree
# #   operator
# #       |
# #      / \
# #    x    n

# for i in range(200):
#     sol = TreeNode(initial_solution.value)
#     sol.left = initial_solution.left
                                
#     # mutation
#     sol.value = operators[random.randint(0,num_operators)]
#     sol.right =  TreeNode(random.randint(1,10))

#     ev = evaluate_tree(sol, {})
#     # print(ev)
#     if mse(evaluate_tree(initial_solution,{}), y[0]) > mse(ev,y[0]):
#         initial_solution.value = sol.value
#         initial_solution.right = sol.right
#         print("found better solution")
#         print(mse(evaluate_tree(sol, {}),y))


# print(evaluate_tree(initial_solution,{}))
# print(f"{initial_solution.left.value} {initial_solution.value} {initial_solution.right.value}")
    

